# FedGen — full-sharing family

## Motivation

### Why full sharing as a comparison?
The partial-sharing family (notebook 03a) is the privacy-preserving configuration relevant to
healthcare deployment. The full-sharing family exists as the **upper-bound reference**: by
averaging the entire model θ = [θ_f; θ_p] each round, clients benefit from a shared feature
extractor. Any accuracy gap between partial and full quantifies the **cost of privacy** — the
information lost by keeping θ_f local.

If partial ≈ full at a given α, the privacy constraint is essentially free.
If partial ≪ full, the feature extractor carries cross-client information that local-only
training cannot recover, and the partial approach needs the generator's inductive bias more.

### Relationship to the original paper
Zhu et al. (2021) Algorithm 1 describes the full-sharing case as the default:
the entire model is uploaded, averaged, and downloaded each round. The generator is an add-on
that regularises local training. Our `fedgen_zhu_full_no_proto` is the closest to this original algorithm.

### Communication cost
Full sharing transmits ~48k params per client per round (feature extractor dominates), making
it ~100× more expensive than partial sharing. In bandwidth-constrained hospital networks,
this is often impractical — but it sets the accuracy ceiling we're trying to approach with
partial sharing.

---

## Variant catalogue

All five variants mirror the partial family row-for-row. The only structural difference is:
- The feature extractor θ_f is **shared and averaged** each round (no persistent local models)
- Evaluation uses the **single global model** end-to-end (no local encoders)


### 1. `fedgen_full` — hard CE + centroid

```
for round t:
  server  → clients : full model θ, generator ω
  each client k:
    θ_k ← θ                                    # full model copy
    sample ŷ ~ U{0,1},  ε ~ N(0,I)
    Ẑ = G_ω(ŷ, ε)                              ← frozen
    for each local epoch:
      step 1 – full model:  L_real = CE_w(θ_k(x), y)
      step 2 – g_r only:    L_kd   = CE(g_r(Ẑ), ŷ)       # hard label
    send θ_k state + per-class latent stats to server
  server:
    θ ← FedAvg(θ_k states, sample counts)
    extract {W_k, b_k} from each θ_k for generator update
    prototypes ← centroid
    generator update:
      min CE(mean_k softmax(W_k·Ẑ' + b_k), ŷ')
        + λ_proto · ‖Ẑ' − z̄_ŷ'‖²
```


### 2. `fedgen_full_medoid` — hard CE + medoid

Same as `fedgen_full` but the prototype anchor is the medoid:
```
  z̄_y = argmin_{z ∈ Z_y} ‖z − mean(Z_y)‖
```


### 3. `fedgen_zhu_full` — KL ensemble + centroid (Zhu Algorithm 1)

```
for round t:
  server  → clients : full model θ, generator ω, {W_k, b_k} from previous round
  each client k:
    θ_k ← θ
    sample ŷ, ε; Ẑ = G_ω(ŷ, ε)
    p_ens = mean_k softmax(W_k·Ẑ + b_k)        ← soft ensemble teacher
    for each local epoch:
      step 1 – full model:  L_real = CE_w(θ_k(x), y)
      step 2 – g_r only:    L_kd   = -Σ p_ens · log softmax(g_r(Ẑ))
    send θ_k to server
  server:
    θ ← FedAvg(θ_k states)
    ensemble teacher ← {W_k, b_k} from this round
    generator update with centroid prototype
```

This is the closest variant to the paper's original Algorithm 1.


### 4. `fedgen_zhu_full_medoid` — KL ensemble + medoid

Same as `fedgen_zhu_full` but medoid anchor.


### 5. `fedgen_zhu_full_no_proto` — KL ensemble + no prototype constraint

```
for round t:
  server  → clients : full model θ, generator ω, {W_k, b_k} from previous round
  each client k:
    (identical to fedgen_zhu_full — KL ensemble KD, full model shared)
  server:
    θ ← FedAvg(θ_k states)
    ensemble teacher ← this round's {W_k, b_k}
    generator update:
      min CE(mean_k softmax(W_k·Ẑ' + b_k), ŷ')     # NO prototype term (λ_proto=0)
```

Full-sharing counterpart. Same ablation question: does the prototype constraint matter when
the feature extractor is already shared and averaged? In the full-sharing case the latent
space is globally aligned each round, so the prototype anchor may be less necessary than in
the partial case where each client's encoder drifts independently.


### 6. `fedgen_gmm_full` — GMM sampler ablation

```
for round t:
  server  → clients : full model θ, GMM params {μ_y, σ_y}
  each client k:
    θ_k ← θ
    Ẑ, ŷ ~ GMMSampler(n_per_class)
    for each local epoch:
      L_real + L_kd (hard CE on GMM samples)
    send θ_k + per-class distribution to server
  server:
    θ ← FedAvg; GMM.update(client distributions)
```

Tests whether GMM's advantage at high heterogeneity (partial family) survives full averaging,
where the shared feature extractor already provides cross-client alignment.

---

## Experimental design

Same grid as the partial family (5 variants × 2 data cases × 5 alphas × 3 seeds).
The key comparisons are:

1. **Within this notebook**: isolate KD loss (hard CE vs KL) and anchor (centroid vs medoid)
   effects when the feature extractor is shared — i.e. when privacy is not a constraint.
2. **Across notebooks (03a vs 03b)**: for each variant pair (e.g. `fedgen_zhu` vs
   `fedgen_zhu_full`), the accuracy gap measures the **privacy–accuracy tradeoff**.
   A small gap validates partial sharing as viable for clinical deployment.
   A large gap motivates more aggressive KD strategies to compensate for the missing
   cross-client feature alignment.

## Setup

In [2]:
import os, sys, json, glob
import numpy as np
import torch

sys.path.insert(0, '..')
from UC1Utils   import ensure_data
from UC1FLUtils import (
    MLP, Generator, GMMSampler,
    load_clients, save_result,
    N_CLIENTS, FL_ROUNDS, LOCAL_EPOCHS, PATIENCE, NOISE_DIM,
)
from UC1FedGenFull import (
    run_fedgen_full,
    run_fedgen_full_medoid,
    run_fedgen_zhu_full,
    run_fedgen_zhu_full_medoid,
    run_fedgen_zhu_full_no_proto,
    run_fedgen_gmm_full,
)
from UC1FedGenFull_pyhat import (
    run_fedgen_zhu_pyhat_full, run_fedgen_zhu_pyhat_full_medoid,
)

SEED = 42
torch.manual_seed(SEED) 
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Paths ──────────────────────────────────────────────────────────────────────
FEDERATED_DIR  = '../federated_data'
FILTERED_DIR   = os.path.join(FEDERATED_DIR, 'filtered')
UNFILTERED_DIR = os.path.join(FEDERATED_DIR, 'unfiltered')
RESULTS_DIR    = 'results'

CENTRALIZED_AUC = 0.658

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('figures',   exist_ok=True)

# ── Shared hyperparameters — loaded from FedAvg tuning ────────────────────────
_hp_path = os.path.join(FEDERATED_DIR, 'fl_hyperparams.json')
if not os.path.exists(_hp_path):
    raise FileNotFoundError(
        f'Hyperparameters not found at {_hp_path}.\n'
        f'Run 02_UC1_Federated.ipynb first.'
    )
with open(_hp_path) as f:
    _hp = json.load(f)

HIDDEN_DIM = _hp['hidden_dim']
DROPOUT    = _hp['dropout']
LR         = _hp['lr']
BATCH_SIZE = _hp['batch_size']

SEEDS       = [42, 123, 7]
ALPHA_SWEEP = [0.1, 0.5, 1.0, 5.0, 10.0]
latent_dim  = HIDDEN_DIM // 4

print(f'hidden_dim={HIDDEN_DIM}  dropout={DROPOUT:.4f}  lr={LR:.6f}  '
      f'batch_size={BATCH_SIZE}  latent_dim={latent_dim}')

/mnt/c/Users/Marc/Documents/Uni/4t/TFG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
hidden_dim=512  dropout=0.1803  lr=0.009296  batch_size=256  latent_dim=128


## Communication cost

Full-sharing transmits the whole model each round, so per-round cost is
dominated by the feature extractor.

In [3]:
_m = MLP(99, HIDDEN_DIM, dropout=DROPOUT)   # 99 = placeholder input_dim
_g = Generator(latent_dim, NOISE_DIM)

n_full      = sum(p.numel() for p in _m.parameters())
n_gen       = sum(p.numel() for p in _g.parameters())
n_gmm_bytes = GMMSampler(latent_dim, 'cpu').param_bytes()
del _m, _g

B = 4
bytes_fedavg_full   = 2 * n_full             * B * N_CLIENTS
bytes_fedgen_neural = (2 * n_full + n_gen)   * B * N_CLIENTS
bytes_fedgen_gmm    = 2 * n_full * B * N_CLIENTS + n_gmm_bytes * N_CLIENTS

print(f'{"Variant":<27} {"MB/round":>10} {"vs FedAvg full":>16}')
print('-' * 56)
for name, b in [
    ('FedAvg full (baseline)',  bytes_fedavg_full),
    ('FedGen full (neural G)',  bytes_fedgen_neural),
    ('FedGen full (GMM)',       bytes_fedgen_gmm),
]:
    ratio = f'{b/bytes_fedavg_full:.3f}×' if 'baseline' not in name else '(baseline)'
    print(f'{name:<27} {b/1024**2:>10.3f} {ratio:>16}')

Variant                       MB/round   vs FedAvg full
--------------------------------------------------------
FedAvg full (baseline)           8.296       (baseline)
FedGen full (neural G)           8.390           1.011×
FedGen full (GMM)                8.306           1.001×


## Run the full 5-variant × 2-case × α-sweep × seed grid

Results land in `results/{data_case}/alpha_{α}/seed_{s}/{variant}.json`.

In [4]:
re_train = False

VARIANTS = {
    'fedgen_full': lambda clients, input_dim, seed: run_fedgen_full(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_full_medoid': lambda clients, input_dim, seed: run_fedgen_full_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_full': lambda clients, input_dim, seed: run_fedgen_zhu_full(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_full_medoid': lambda clients, input_dim, seed: run_fedgen_zhu_full_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_gmm_full': lambda clients, input_dim, seed: run_fedgen_gmm_full(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE,
    ),
    'fedgen_zhu_full_no_proto': lambda clients, input_dim, seed: run_fedgen_zhu_full_no_proto(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    )
}

for data_case, data_dir in [('filtered', FILTERED_DIR), ('unfiltered', UNFILTERED_DIR)]:
    print(f'\n{"#"*60}\nData case: {data_case}\n{"#"*60}')

    for alpha in ALPHA_SWEEP:
        for seed in SEEDS:
            r_base = os.path.join(RESULTS_DIR, data_case)
            r_dir  = os.path.join(r_base, f'alpha_{alpha}', f'seed_{seed}')

            try:
                clients = load_clients(alpha, data_dir, N_CLIENTS)
            except (FileNotFoundError, ValueError) as e:
                print(f'  Skipping α={alpha} [{data_case}]: {e}')
                continue

            input_dim = clients[0]['X_train'].shape[1]

            for vname, run_fn in VARIANTS.items():
                out_path = os.path.join(r_dir, f'{vname}.json')
                if os.path.exists(out_path) and not re_train:
                    print(f'  [{data_case}] α={alpha} seed={seed} {vname}: already done, skipping.')
                    continue

                print(f'\n{"="*60}\n{vname}  [{data_case}]  α={alpha}  seed={seed}\n{"="*60}')
                auc, pc, hist, cmb = run_fn(clients, input_dim, seed)
                save_result(r_base, alpha, seed, vname, auc, pc, hist, cmb)

print('\nAll full-family experiments complete.')


############################################################
Data case: filtered
############################################################
  client_0: 28,959 train  pos_rate=6.8%  n_pos=1977
  client_1: 2,472 train  pos_rate=43.7%  n_pos=1080
  client_2: 22,830 train  pos_rate=5.0%  n_pos=1141
  client_3: 6,799 train  pos_rate=43.2%  n_pos=2939
  client_4: 5,401 train  pos_rate=9.0%  n_pos=488
  [filtered] α=0.1 seed=42 fedgen_full: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_full_medoid: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu_full: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu_full_medoid: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_gmm_full: already done, skipping.
  [filtered] α=0.1 seed=42 fedgen_zhu_full_no_proto: already done, skipping.
  client_0: 28,959 train  pos_rate=6.8%  n_pos=1977
  client_1: 2,472 train  pos_rate=43.7%  n_pos=1080
  client_2: 22,830 train  pos_rate=5.0%  n_pos=1141
  client_3: 6,799 tr

## Results summary (full family only)

In [5]:
FULL_NAMES = {
    'fedgen_full', 'fedgen_full_medoid',
    'fedgen_zhu_full', 'fedgen_zhu_full_medoid',
    'fedgen_zhu_full_no_proto',
    'fedgen_gmm_full',
}

print(f'{"Case":<12} {"Alpha":>6}  {"Seed":>6}  {"Variant":<26}  '
      f'{"Test AUC":>9}  {"Rounds":>7}  {"Final MB":>9}')
print('-' * 88)

for path in sorted(glob.glob(f'{RESULTS_DIR}/*/*/seed_*/*.json')):
    parts = path.replace(RESULTS_DIR + '/', '').replace('.json', '').split('/')
    case, alpha_s, seed_s, variant = parts
    if variant not in FULL_NAMES:
        continue
    r = json.load(open(path))
    final_mb = r['cumul_mb'][-1] if r['cumul_mb'] else float('nan')
    print(f'{case:<12} {alpha_s.replace("alpha_",""):>6}  '
          f'{seed_s.replace("seed_",""):>6}  '
          f'{variant:<26}  '
          f'{r["test_auc"]:>9.4f}  '
          f'{len(r["history"]):>7}  '
          f'{final_mb:>9.2f}')

Case          Alpha    Seed  Variant                      Test AUC   Rounds   Final MB
----------------------------------------------------------------------------------------
filtered        0.1     123  fedgen_full                    0.5819       11      92.28
filtered        0.1     123  fedgen_full_medoid             0.5813        9      75.51
filtered        0.1     123  fedgen_gmm_full                0.5830       18     149.50
filtered        0.1     123  fedgen_zhu_full                0.5837       24     201.35
filtered        0.1     123  fedgen_zhu_full_medoid         0.5854       16     134.23
filtered        0.1     123  fedgen_zhu_full_no_proto       0.5835       17     142.62
filtered        0.1      42  fedgen_full                    0.5825        9      75.51
filtered        0.1      42  fedgen_full_medoid             0.5867       24     201.35
filtered        0.1      42  fedgen_gmm_full                0.5801       11      91.36
filtered        0.1      42  fedgen_zhu_f

In [6]:
re_train = False

VARIANTS_PYHAT = {
    'fedgen_zhu_pyhat_full': lambda clients, input_dim, seed: run_fedgen_zhu_pyhat_full(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_zhu_pyhat_full_medoid': lambda clients, input_dim, seed: run_fedgen_zhu_pyhat_full_medoid(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
}
for data_case, data_dir in [('unfiltered', UNFILTERED_DIR)]:
    for alpha in ALPHA_SWEEP:
        print(f'\n{"="*60}\n  α = {alpha}\n{"="*60}')
        try:
            clients = load_clients(alpha, data_dir, N_CLIENTS)
        except (FileNotFoundError, ValueError) as e:
            print(f'  Skipping α={alpha} [{data_case}]: {e}')
            continue
        input_dim = clients[0]['X_train'].shape[1]

        for seed in SEEDS:
            print(f'\n  ── seed={seed} ──')
            for vname, vfn in VARIANTS_PYHAT.items():
                out_path = os.path.join(RESULTS_DIR, data_case,
                                        f'alpha_{alpha}', f'seed_{seed}', f'{vname}.json')
                if os.path.exists(out_path) and not re_train:
                    print(f'  [{vname}] already exists, skipping.')
                    continue

                print(f'  ▶ {vname}')
                auc, per_client, hist, mb = vfn(clients, input_dim, seed)
                save_result(os.path.join(RESULTS_DIR, data_case),
                            alpha, seed, vname, auc, per_client, hist, mb)


  α = 0.1
  client_0: 2 train  pos_rate=0.0%  n_pos=0
  Skipping α=0.1 [unfiltered]: client_0 at α=0.1 has zero positive training examples. Regenerate the partition with a higher min_pos_abs.

  α = 0.5
  client_0: 11,874 train  pos_rate=1.1%  n_pos=126
  client_1: 4,709 train  pos_rate=18.4%  n_pos=865
  client_2: 2,652 train  pos_rate=45.4%  n_pos=1203
  client_3: 22,704 train  pos_rate=0.1%  n_pos=21
  client_4: 24,591 train  pos_rate=22.1%  n_pos=5431

  ── seed=42 ──
  [fedgen_zhu_pyhat_full] already exists, skipping.
  [fedgen_zhu_pyhat_full_medoid] already exists, skipping.

  ── seed=123 ──
  [fedgen_zhu_pyhat_full] already exists, skipping.
  [fedgen_zhu_pyhat_full_medoid] already exists, skipping.

  ── seed=7 ──
  [fedgen_zhu_pyhat_full] already exists, skipping.
  [fedgen_zhu_pyhat_full_medoid] already exists, skipping.

  α = 1.0
  client_0: 21,461 train  pos_rate=0.0%  n_pos=8
  client_1: 3,512 train  pos_rate=16.6%  n_pos=582
  client_2: 15,745 train  pos_rate=11.7%  n_

In [7]:
from UC1FLUtils import (
    load_clients, save_result,
    N_CLIENTS, FL_ROUNDS, LOCAL_EPOCHS, PATIENCE, NOISE_DIM,
)
from UC1FedGenFull_paper import (
    run_fedgen_paper_full, run_fedgen_paper_full_no_proto,
)

In [8]:
re_train = False

VARIANTS_PAPER_FULL = {
    'fedgen_paper_full': lambda clients, input_dim, seed: run_fedgen_paper_full(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
    'fedgen_paper_full_no_proto': lambda clients, input_dim, seed: run_fedgen_paper_full_no_proto(
        clients, input_dim, seed,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT, lr=LR, batch_size=BATCH_SIZE,
        n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
        local_epochs=LOCAL_EPOCHS, patience=PATIENCE, noise_dim=NOISE_DIM,
    ),
}

for data_case, data_dir in [('filtered', FILTERED_DIR), ('unfiltered', UNFILTERED_DIR)]:
    for alpha in ALPHA_SWEEP:
        print(f'\n{"="*60}\n  {data_case} α = {alpha}\n{"="*60}')
        try:
            clients = load_clients(alpha, data_dir, N_CLIENTS)
        except (FileNotFoundError, ValueError) as e:
            print(f'  Skipping α={alpha} [{data_case}]: {e}')
            continue
        input_dim = clients[0]['X_train'].shape[1]

        for seed in SEEDS:
            print(f'\n  ── seed={seed} ──')
            for vname, vfn in VARIANTS_PAPER_FULL.items():
                out_path = os.path.join(RESULTS_DIR, data_case,
                                        f'alpha_{alpha}', f'seed_{seed}', f'{vname}.json')
                if os.path.exists(out_path) and not re_train:
                    print(f'  [{vname}] already exists, skipping.')
                    continue

                print(f'  ▶ {vname}')
                auc, per_client, hist, mb = vfn(clients, input_dim, seed)
                save_result(os.path.join(RESULTS_DIR, data_case),
                            alpha, seed, vname, auc, per_client, hist, mb)


  filtered α = 0.1
  client_0: 28,959 train  pos_rate=6.8%  n_pos=1977
  client_1: 2,472 train  pos_rate=43.7%  n_pos=1080
  client_2: 22,830 train  pos_rate=5.0%  n_pos=1141
  client_3: 6,799 train  pos_rate=43.2%  n_pos=2939
  client_4: 5,401 train  pos_rate=9.0%  n_pos=488

  ── seed=42 ──
  ▶ fedgen_paper_full
  [fedgen_paper_full] R01: val=0.5819 test=0.5756 p̂=['0.885', '0.115'] cumul=8.39MB
  [fedgen_paper_full] R02: val=0.5920 test=0.5834 p̂=['0.885', '0.115'] cumul=16.78MB
  [fedgen_paper_full] R03: val=0.5916 test=0.5819 p̂=['0.885', '0.115'] cumul=25.17MB
  [fedgen_paper_full] R04: val=0.5944 test=0.5839 p̂=['0.885', '0.115'] cumul=33.56MB
  [fedgen_paper_full] R05: val=0.5962 test=0.5835 p̂=['0.885', '0.115'] cumul=41.95MB
  [fedgen_paper_full] R06: val=0.5952 test=0.5833 p̂=['0.885', '0.115'] cumul=50.34MB
  [fedgen_paper_full] R07: val=0.5968 test=0.5843 p̂=['0.885', '0.115'] cumul=58.73MB
  [fedgen_paper_full] R08: val=0.6002 test=0.5866 p̂=['0.885', '0.115'] cumul=67.1